# Análisis Estadístico de Features vs. Target

En este notebook se evalúa la asociación de cada feature (original + generada en `00_feat_engineer_initial.ipynb`) con el target `fraude`, mediante pruebas estadísticas específicas para variables continuas y para variables categóricas. El objetivo es tener evidencia cuantitativa (no solo intuición del EDA) para priorizar qué features llevar al modelado.


# Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd().parent if (Path.cwd().name == "02_feature_engineer") else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.feature_engineer_utils.statistical_tests import FeatureStatTests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


# Configs

In [2]:
DATASET_DIR = ROOT_DIR / "dataset"
TRAIN_FEAT_PATH = DATASET_DIR / "train_feat_engineer.csv"

OUTPUT_DIR = ROOT_DIR / "02_feature_engineer" / "stats_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTINUOUS_STATS_PATH = OUTPUT_DIR / "stats_fraud_continuos.csv"
CATEGORICAL_STATS_PATH = OUTPUT_DIR / "stats_fraud_categorical.csv"
WOE_BINS_PATH = OUTPUT_DIR / "woe_bins_categorical.csv"

TARGET_COL = "fraude"
IV_MAX_BINS = 20
LG_MAX_CATEGORIES = 20

TRAIN_FEAT_PATH


PosixPath('/Users/dpiedrahita/proyectos/DS_pro/dataset/train_feat_engineer.csv')

# Read Data

In [3]:
df = pd.read_csv(TRAIN_FEAT_PATH, low_memory=False)
print(f"Shape: {df.shape}")
df.head()

Shape: (104981, 80)


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude,a_freq_enc,a_grouped,b_missing,b_is_zero,b_log,b_top1pct,b_bin,c_missing,c_log,c_top1pct,c_bin,d_missing,d_is_zero,d_log,d_top1pct,d_bin,e_is_zero,e_log,e_top1pct,e_bin,f_missing,f_is_zero,f_log,f_top1pct,f_bin,g_missing,g_freq_enc,g_grouped,h_is_zero,h_log,h_top1pct,h_bin,j_freq_enc,j_grouped,j_target_enc,l_missing,l_is_zero,l_log,l_top1pct,l_bin,m_missing,m_is_zero,m_log,m_top1pct,m_bin,o_freq_enc,o_target_enc,fecha_hour,fecha_weekday,fecha_is_weekend,fecha_hour_sin,fecha_hour_cos,monto_log,monto_top1pct,monto_bin,score_is_zero,score_log,score_top1pct,score_bin,pca_dm_1,ratio_monto_l
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,0.937548,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25,0,0.850821,4,0,0,0.553195,0,2,0,8.750762,0,1,0,0,2.708050,0,2,0,0.130395,0,0,0,0,3.218876,0,3,0,0.755489,BR,0,2.079442,0,1,0.004563,__OTHER__,0.042624,0,0,7.767264,0,2,0,0,6.093570,0,3,0.724388,0.051838,0,6,1,0.0,1.0,3.143290,0,2,0,3.258097,0,1,142.485292,0.009394
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,0.791998,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7,0,0.850821,4,0,0,0.562355,0,2,0,9.960439,0,1,0,0,3.044522,0,2,0,0.415293,0,2,0,0,2.079442,0,2,0,0.755489,BR,0,1.098612,0,0,0.000133,__OTHER__,0.023563,0,0,7.751475,0,2,0,0,4.304065,0,1,0.724388,0.051838,0,6,1,0.0,1.0,1.945910,0,0,0,2.079442,0,0,-226.040600,0.002582
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,0.688592,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91,1,0.850821,4,0,0,0.643221,0,4,0,8.297501,0,0,0,0,3.931826,1,3,0,0.242292,0,1,0,0,0.693147,0,0,0,0.755489,BR,0,1.386294,0,1,0.002648,__OTHER__,0.044569,0,0,5.463832,0,0,0,0,5.451038,0,2,0.113421,0.230557,0,6,1,0.0,1.0,3.320349,0,2,0,4.521789,0,4,-66.040159,0.113489
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,0.654161,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91,1,0.850821,4,0,0,0.603496,0,4,0,11.509057,0,3,0,0,0.693147,0,0,1,0.000000,0,0,0,0,1.609438,0,1,0,0.755489,BR,0,3.367296,0,4,0.000438,__OTHER__,0.119965,0,0,6.490724,0,0,0,1,0.000000,0,0,0.113421,0.228374,0,6,1,0.0,1.0,3.713816,0,3,0,4.521789,0,4,-299.693151,0.060805
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,0.532994,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93,0,0.101837,2,0,0,0.469504,0,0,0,10.887948,0,2,0,0,1.098612,0,0,1,0.000000,0,0,0,0,5.579730,0,4,0,0.204132,AR,0,3.555348,0,4,0.000076,__OTHER__,0.032399,0,0,7.783641,0,2,0,0,2.397895,0,0,0.724388,0.051838,0,6,1,0.0,1.0,1.981001,0,0,0,4.543295,0,4,-289.663017,0.002604


## Clasificación de variables: continuas vs. categóricas

Se excluyen del análisis: `k` (ID, 100% único), `fecha` (timestamp crudo, ya se extrajeron sus features derivadas) y `fraude` (el target).

El resto se clasifica automáticamente:
- **Categóricas**: las categóricas originales (`a`, `g`, `j`, `n`, `o`, `p`), `fecha_weekday` (día de semana, nominal), y cualquier feature generada cuyo sufijo indique que es binaria/discreta (`_missing`, `_is_zero`, `_top1pct`, `_is_weekend`, `_bin`, `_grouped`).
- **Continuas**: todo lo demás (numéricas originales, escalas logarítmicas, encodings de frecuencia/target, PCA, ratios, componentes cíclicos de hora).


In [4]:
EXCLUDE_COLS = {"k", "fecha", TARGET_COL}
CATEGORICAL_SUFFIXES = ("_missing", "_is_zero", "_top1pct", "_is_weekend", "_bin", "_grouped")
EXTRA_CATEGORICAL_COLS = {"a", "g", "j", "n", "o", "p", "fecha_weekday"}

categorical_cols = sorted(
    c for c in df.columns
    if c not in EXCLUDE_COLS
    and (c in EXTRA_CATEGORICAL_COLS or c.endswith(CATEGORICAL_SUFFIXES))
)
continuous_cols = sorted(
    c for c in df.columns
    if c not in EXCLUDE_COLS and c not in categorical_cols
)

print(f"Variables continuas ({len(continuous_cols)}): {continuous_cols}")
print()
print(f"Variables categóricas ({len(categorical_cols)}): {categorical_cols}")


Variables continuas (31): ['a_freq_enc', 'b', 'b_log', 'c', 'c_log', 'd', 'd_log', 'e', 'e_log', 'f', 'f_log', 'fecha_hour', 'fecha_hour_cos', 'fecha_hour_sin', 'g_freq_enc', 'h', 'h_log', 'j_freq_enc', 'j_target_enc', 'l', 'l_log', 'm', 'm_log', 'monto', 'monto_log', 'o_freq_enc', 'o_target_enc', 'pca_dm_1', 'ratio_monto_l', 'score', 'score_log']

Variables categóricas (46): ['a', 'a_grouped', 'b_bin', 'b_is_zero', 'b_missing', 'b_top1pct', 'c_bin', 'c_missing', 'c_top1pct', 'd_bin', 'd_is_zero', 'd_missing', 'd_top1pct', 'e_bin', 'e_is_zero', 'e_top1pct', 'f_bin', 'f_is_zero', 'f_missing', 'f_top1pct', 'fecha_is_weekend', 'fecha_weekday', 'g', 'g_grouped', 'g_missing', 'h_bin', 'h_is_zero', 'h_top1pct', 'j', 'j_grouped', 'l_bin', 'l_is_zero', 'l_missing', 'l_top1pct', 'm_bin', 'm_is_zero', 'm_missing', 'm_top1pct', 'monto_bin', 'monto_top1pct', 'n', 'o', 'p', 'score_bin', 'score_is_zero', 'score_top1pct']


## Pruebas para variables continuas

Por cada variable continua se corren las siguientes pruebas:

1. **Mann–Whitney U** — compara la distribución de la variable entre fraude=0 y fraude=1 sin asumir normalidad. Estadístico + p-value.
2. **KS Statistic (Kolmogorov–Smirnov)** — máxima distancia entre las CDF de ambos grupos. A mayor KS, mejor separación.
3. **Correlación Punto-Biserial** — correlación entre la variable continua y el target binario (equivalente a Pearson). Indica dirección y fuerza.
4. **ROC-AUC univariado** — usa la variable sola como score; se reporta `auc_abs = max(auc, 1-auc)` para que la dirección no afecte la magnitud.
5. **d de Cohen** — tamaño del efecto: diferencia de medias entre los dos grupos dividida por la desviación estándar combinada (pooled).

La implementación además corre una sexta prueba, complementaria a las anteriores:

6. **Wald sobre regresión logística univariada** se ajusta una regresión logística; reporta coeficiente (β), odds ratio y su p-value, y detecta separación perfecta/inestabilidad numérica del optimizador.

**Interpretación embebida:**
- AUC: <0.6 "casi aleatorio", 0.6–0.7 "separación débil", 0.7–0.8 "moderada", ≥0.8 "fuerte".
- d de Cohen: <0.2 despreciable, <0.5 pequeño, <0.8 mediano, ≥0.8 grande.


In [5]:
fts = FeatureStatTests(df)

continuous_results = [fts.compute(col, TARGET_COL, kind="continuous") for col in continuous_cols]
stats_fraud_continuos = pd.DataFrame(continuous_results).sort_values("auc_abs", ascending=False).reset_index(drop=True)

stats_fraud_continuos.to_csv(CONTINUOUS_STATS_PATH, index=False)
print(f"Guardado: {CONTINUOUS_STATS_PATH}")
stats_fraud_continuos


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))


Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/stats_outputs/stats_fraud_continuos.csv


,variable,mannwhitney_stat,mannwhitney_p_value,ks_stat,ks_p_value,point_biserial_corr,point_biserial_p_value,auc,auc_abs,auc_interpretation,cohens_d,cohens_d_interpretation,logit_beta,logit_odds_ratio,logit_p_value,logit_separation_detected
0,o_freq_enc,409488073.0,0.000000e+00,0.450808,0.000000e+00,-0.232615,0.000000e+00,0.244056,0.755944,separación moderada,-1.078816,grande,-3.478826e+00,3.084359e-02,0.000000e+00,False
1,o_target_enc,132635438.5,0.000000e+00,0.450808,0.000000e+00,0.289675,0.000000e+00,0.755146,0.755146,separación moderada,1.365126,grande,1.334375e+01,6.239048e+05,0.000000e+00,False
2,score_log,148621661.0,0.000000e+00,0.421760,0.000000e+00,0.098848,3.656043e-226,0.725634,0.725634,separación moderada,0.448055,pequeño,7.284718e-01,2.071912e+00,1.602985e-213,False
3,score,148621661.0,0.000000e+00,0.421760,0.000000e+00,0.170763,0.000000e+00,0.725634,0.725634,separación moderada,0.781717,mediano,3.022453e-02,1.030686e+00,0.000000e+00,False
4,f_log,363092386.5,0.000000e+00,0.281655,0.000000e+00,-0.139117,0.000000e+00,0.313106,0.686894,separación débil,-0.635976,mediano,-4.442609e-01,6.412981e-01,0.000000e+00,False
5,f,371816724.5,0.000000e+00,0.282485,0.000000e+00,-0.011974,1.046492e-04,0.313524,0.686476,separación débil,-0.054011,despreciable,-6.710215e-03,9.933122e-01,1.450316e-62,False
6,l_log,365856118.0,0.000000e+00,0.255437,2.560702e-297,-0.182159,0.000000e+00,0.324529,0.675471,separación débil,-0.835580,grande,-3.329196e-01,7.168278e-01,0.000000e+00,False
7,l,365856118.0,0.000000e+00,0.255437,2.560702e-297,-0.119141,0.000000e+00,0.324529,0.675471,separación débil,-0.541223,mediano,-4.044191e-04,9.995957e-01,2.422310e-305,False
8,ratio_monto_l,169925590.0,0.000000e+00,0.246069,1.409003e-258,0.055019,3.016679e-70,0.660961,0.660961,separación débil,0.254885,pequeño,2.365680e-02,1.023939e+00,8.802008e-22,False
9,m_log,354917306.0,0.000000e+00,0.233943,1.342863e-247,-0.138970,0.000000e+00,0.339605,0.660395,separación débil,-0.633662,mediano,-2.473061e-01,7.809016e-01,0.000000e+00,False


## Pruebas para variables categóricas

Por cada variable categórica se corren las siguientes pruebas:

1. **Chi-Cuadrado de independencia** — tabla de contingencia variable × target; prueba si hay asociación estadísticamente significativa.
2. **V de Cramér** — tamaño del efecto asociado al Chi², normalizado entre 0 y 1, para saber qué tan fuerte es la asociación (no solo si es significativa).
3. **Information Value (IV) con Weight of Evidence (WoE)** — por cada categoría, WoE = ln(%buenos / %malos); el IV es la suma ponderada de esos WoE. Con `iv_max_bins=20` se agrupan categorías raras en `"__OTHER__"` antes de calcular (evita explosión en `j`, que tiene ~8,250 valores).

La implementación además corre una cuarta prueba, complementaria a las anteriores:

4. **Regresión logística con dummies + likelihood-ratio test** regresión logística, codifica la variable con one-hot (colapsando categorías raras si hay más de `lg_max_categories=20`), ajusta un modelo logístico completo vs. uno nulo, y compara ambos con un test de razón de verosimilitud (chi-cuadrado).

Además se calcula un quinto indicador, que codifica la variable con su propio Weight of Evidence (WoE) por categoría y calcula el AUC sobre ese score continuo. Esto da un `auc woe abs` en la misma escala 0.5-1 que el `auc abs` de las variables continuas, lo que permite comparar directamente el poder predictivo de una categórica contra el de una continua (se usa en la sección de comparación más adelante).

**Interpretación embebida:**
- IV: <0.02 inútil, <0.10 débil, <0.30 medio, <0.50 fuerte, ≥0.50 "sospechoso, revisar leakage".
- V de Cramér: <0.1 despreciable, <0.3 débil, <0.5 moderada, ≥0.5 fuerte.


In [6]:
categorical_results = [
    fts.compute(col, TARGET_COL, kind="categorical", iv_max_bins=IV_MAX_BINS, lg_max_categories=LG_MAX_CATEGORIES)
    for col in categorical_cols
]
stats_fraud_categorical = pd.DataFrame(categorical_results).sort_values("iv", ascending=False).reset_index(drop=True)

stats_fraud_categorical.to_csv(CATEGORICAL_STATS_PATH, index=False)
print(f"Guardado: {CATEGORICAL_STATS_PATH}")
stats_fraud_categorical


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.p

Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/stats_outputs/stats_fraud_categorical.csv


,variable,chi2_stat,chi2_p_value,chi2_dof,cramers_v,cramers_v_interpretation,iv,iv_interpretation,auc_woe,auc_woe_abs,auc_woe_interpretation,lr_stat,lr_p_value,lr_dof,lr_separation_detected
0,o,9049.180877,0.000000e+00,2,0.293595,débil,1.114077,"sospechoso, revisar leakage",0.244056,0.755944,separación moderada,6175.151885,0.000000e+00,2,False
1,score_bin,5633.865898,0.000000e+00,4,0.231658,débil,0.958800,"sospechoso, revisar leakage",0.247815,0.752185,separación moderada,4804.698363,0.000000e+00,4,False
2,f_bin,2284.752268,0.000000e+00,5,0.147525,débil,0.428635,fuerte,0.323014,0.676986,separación débil,2168.161065,0.000000e+00,5,True
3,n,3074.409895,0.000000e+00,1,0.171130,débil,0.362648,fuerte,0.386316,0.613684,separación débil,2118.361084,0.000000e+00,1,False
4,l_bin,2106.181662,0.000000e+00,5,0.141642,débil,0.353298,fuerte,0.338970,0.661030,separación débil,1865.445028,0.000000e+00,5,True
5,m_bin,1849.277546,0.000000e+00,5,0.132723,débil,0.319079,fuerte,0.347260,0.652740,separación débil,1667.636483,0.000000e+00,5,False
6,f_is_zero,1968.530317,0.000000e+00,1,0.136935,débil,0.283142,medio,0.384899,0.615101,separación débil,1578.811537,0.000000e+00,1,False
7,p,1285.352763,1.723001e-281,1,0.110651,débil,0.254656,medio,0.375977,0.624023,separación débil,1288.801323,3.068029e-282,1,False
8,m_is_zero,1237.747408,3.818029e-271,1,0.108583,débil,0.168042,medio,0.424650,0.575350,casi aleatorio,955.609643,8.004773e-210,1,False
9,score_top1pct,1825.234515,0.000000e+00,1,0.131857,débil,0.150524,medio,0.464017,0.535983,casi aleatorio,945.523930,1.246617e-207,1,False


## Extracción de tablas WoE por categoría

A continuación, se recorre cada variable categórica llamando a para extraer la tabla completa de WoE por categoría (no solo el IV agregado) y las apila todas en un único archivo.


In [7]:
woe_tables = []
for col in categorical_cols:
    result = fts.compute(
        col, TARGET_COL, kind="categorical",
        iv_max_bins=IV_MAX_BINS, lg_max_categories=LG_MAX_CATEGORIES,
        include_woe_table=True,
    )
    woe_tables.append(result["woe_table"])

woe_bins_categorical = pd.concat(woe_tables, ignore_index=True)
woe_bins_categorical.to_csv(WOE_BINS_PATH, index=False)
print(f"Guardado: {WOE_BINS_PATH}")
woe_bins_categorical


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/statsmodels/base/model.p

Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/stats_outputs/woe_bins_categorical.csv


,variable,categoria,n_total,n_good,n_bad,pct_good,pct_bad,woe,iv_contribution
0,a,1,2885,2627,258,0.026396,0.047483,-0.587163,0.012382
1,a,2,10691,9880,811,0.099261,0.149063,-0.406622,0.020251
2,a,3,2085,1890,195,0.018992,0.035911,-0.637019,0.010778
3,a,4,89320,85142,4178,0.855351,0.767542,0.108318,0.009511
4,a_grouped,1,2885,2627,258,0.026396,0.047483,-0.587163,0.012382
...,...,...,...,...,...,...,...,...,...
171,score_bin,4,20648,17504,3144,0.175851,0.577555,-1.189166,0.477693
172,score_is_zero,0,101667,96319,5348,0.967646,0.982638,-0.015375,0.000230
173,score_is_zero,1,3314,3220,94,0.032354,0.017362,0.622463,0.009332
174,score_top1pct,0,103425,98435,4990,0.988904,0.916866,0.075636,0.005449


## Conclusiones

In [8]:
print("Top 10 continuas por AUC:")
display(stats_fraud_continuos[["variable", "auc_abs", "auc_interpretation", "cohens_d", "cohens_d_interpretation"]].head(10))

print("\nTop 10 categóricas por IV:")
display(stats_fraud_categorical[["variable", "iv", "iv_interpretation", "cramers_v", "cramers_v_interpretation"]].head(10))


Top 10 continuas por AUC:


,variable,auc_abs,auc_interpretation,cohens_d,cohens_d_interpretation
0,o_freq_enc,0.755944,separación moderada,-1.078816,grande
1,o_target_enc,0.755146,separación moderada,1.365126,grande
2,score_log,0.725634,separación moderada,0.448055,pequeño
3,score,0.725634,separación moderada,0.781717,mediano
4,f_log,0.686894,separación débil,-0.635976,mediano
5,f,0.686476,separación débil,-0.054011,despreciable
6,l_log,0.675471,separación débil,-0.835580,grande
7,l,0.675471,separación débil,-0.541223,mediano
8,ratio_monto_l,0.660961,separación débil,0.254885,pequeño
9,m_log,0.660395,separación débil,-0.633662,mediano



Top 10 categóricas por IV:


,variable,iv,iv_interpretation,cramers_v,cramers_v_interpretation
0,o,1.114077,"sospechoso, revisar leakage",0.293595,débil
1,score_bin,0.958800,"sospechoso, revisar leakage",0.231658,débil
2,f_bin,0.428635,fuerte,0.147525,débil
3,n,0.362648,fuerte,0.171130,débil
4,l_bin,0.353298,fuerte,0.141642,débil
5,m_bin,0.319079,fuerte,0.132723,débil
6,f_is_zero,0.283142,medio,0.136935,débil
7,p,0.254656,medio,0.110651,débil
8,m_is_zero,0.168042,medio,0.108583,débil
9,score_top1pct,0.150524,medio,0.131857,débil


## Comparación: features originales vs. features nuevas

¿Las features generadas en `00_feat_engineer_initial.ipynb` mejoran la relación con el target frente a la variable original de la que provienen? 

- Para responder esto de forma comparable entre continuas y categóricas, se usa una única métrica en escala 0.5–1: `auc_abs` para las continuas y (AUC sobre el WoE de la variable) para las categóricas. Ambas responden la misma pregunta: *dado un par (transacción fraudulenta, transacción legítima) al azar, ¿con qué probabilidad la variable los ordena correctamente?*

Se mapea cada feature nueva a la columna original de la que se derivó (excluyendo combinaciones de más de una columna como `pca_dm_1` o `ratio_monto_l`, y las derivadas de `fecha`, que no tiene una métrica base propia porque el timestamp crudo no se evaluó directamente).

In [9]:
FEATURE_ORIGIN_MAP = {
    "a_freq_enc": "a", "a_grouped": "a",
    "b_missing": "b", "b_is_zero": "b", "b_log": "b", "b_top1pct": "b", "b_bin": "b",
    "c_missing": "c", "c_log": "c", "c_top1pct": "c", "c_bin": "c",
    "d_missing": "d", "d_is_zero": "d", "d_log": "d", "d_top1pct": "d", "d_bin": "d",
    "e_is_zero": "e", "e_log": "e", "e_top1pct": "e", "e_bin": "e",
    "f_missing": "f", "f_is_zero": "f", "f_log": "f", "f_top1pct": "f", "f_bin": "f",
    "g_missing": "g", "g_freq_enc": "g", "g_grouped": "g",
    "h_is_zero": "h", "h_log": "h", "h_top1pct": "h", "h_bin": "h",
    "j_freq_enc": "j", "j_grouped": "j", "j_target_enc": "j",
    "l_missing": "l", "l_is_zero": "l", "l_log": "l", "l_top1pct": "l", "l_bin": "l",
    "m_missing": "m", "m_is_zero": "m", "m_log": "m", "m_top1pct": "m", "m_bin": "m",
    "o_freq_enc": "o", "o_target_enc": "o",
    "monto_log": "monto", "monto_top1pct": "monto", "monto_bin": "monto",
    "score_is_zero": "score", "score_log": "score", "score_top1pct": "score", "score_bin": "score",
}

# métrica unificada (auc_abs para continuas, auc_woe_abs para categóricas) por variable
auc_like = pd.concat([
    stats_fraud_continuos.set_index("variable")["auc_abs"],
    stats_fraud_categorical.set_index("variable")["auc_woe_abs"].rename("auc_abs"),
])

comparison_rows = []
for new_col, base_col in FEATURE_ORIGIN_MAP.items():
    if new_col not in auc_like.index or base_col not in auc_like.index:
        continue
    base_metric = auc_like[base_col]
    new_metric = auc_like[new_col]
    comparison_rows.append({
        "feature_original": base_col,
        "feature_nueva": new_col,
        "auc_abs_original": base_metric,
        "auc_abs_nueva": new_metric,
        "delta": new_metric - base_metric,
        "mejora": new_metric > base_metric,
    })

feature_comparison = pd.DataFrame(comparison_rows).sort_values("delta", ascending=False).reset_index(drop=True)
feature_comparison


,feature_original,feature_nueva,auc_abs_original,auc_abs_nueva,delta,mejora
0,j,j_target_enc,0.543508,0.642816,0.099309,True
1,c,c_bin,0.510332,0.540862,0.030529,True
2,score,score_bin,0.725634,0.752185,0.026551,True
3,j,j_freq_enc,0.543508,0.565071,0.021563,True
4,e,e_bin,0.528351,0.531593,0.003242,True
5,c,c_missing,0.510332,0.510950,0.000617,True
6,f,f_log,0.686476,0.686894,0.000418,True
7,e,e_log,0.528351,0.528351,0.000000,False
8,score,score_log,0.725634,0.725634,0.000000,False
9,monto,monto_log,0.559277,0.559277,0.000000,False


In [10]:
n_mejoras = int(feature_comparison["mejora"].sum())
n_total = len(feature_comparison)
print(f"Features nuevas con mejor auc_abs que su original: {n_mejoras}/{n_total} ({n_mejoras/n_total:.1%})")

print("\nTop 10 mejoras (mayor delta):")
display(feature_comparison.head(10))

print("\nTop 10 sin mejora o con retroceso (menor delta):")
display(feature_comparison.tail(10))


Features nuevas con mejor auc_abs que su original: 7/54 (13.0%)

Top 10 mejoras (mayor delta):


,feature_original,feature_nueva,auc_abs_original,auc_abs_nueva,delta,mejora
0,j,j_target_enc,0.543508,0.642816,0.099309,True
1,c,c_bin,0.510332,0.540862,0.030529,True
2,score,score_bin,0.725634,0.752185,0.026551,True
3,j,j_freq_enc,0.543508,0.565071,0.021563,True
4,e,e_bin,0.528351,0.531593,0.003242,True
5,c,c_missing,0.510332,0.510950,0.000617,True
6,f,f_log,0.686476,0.686894,0.000418,True
7,e,e_log,0.528351,0.528351,0.000000,False
8,score,score_log,0.725634,0.725634,0.000000,False
9,monto,monto_log,0.559277,0.559277,0.000000,False



Top 10 sin mejora o con retroceso (menor delta):


,feature_original,feature_nueva,auc_abs_original,auc_abs_nueva,delta,mejora
44,d,d_missing,0.614884,0.501206,-0.113677,False
45,l,l_is_zero,0.675471,0.527011,-0.148459,False
46,m,m_top1pct,0.660395,0.503744,-0.156651,False
47,m,m_missing,0.660395,0.501206,-0.159189,False
48,l,l_top1pct,0.675471,0.504402,-0.171068,False
49,l,l_missing,0.675471,0.500055,-0.175415,False
50,f,f_top1pct,0.686476,0.503841,-0.182635,False
51,f,f_missing,0.686476,0.500055,-0.186420,False
52,score,score_top1pct,0.725634,0.535983,-0.189651,False
53,score,score_is_zero,0.725634,0.507538,-0.218096,False


In [11]:
print("Mejor transformación por variable original (mayor auc_abs_nueva dentro de cada grupo):")
best_per_original = (
    feature_comparison
    .sort_values("auc_abs_nueva", ascending=False)
    .groupby("feature_original", as_index=False)
    .first()
    .sort_values("delta", ascending=False)
)
best_per_original


Mejor transformación por variable original (mayor auc_abs_nueva dentro de cada grupo):


,feature_original,feature_nueva,auc_abs_original,auc_abs_nueva,delta,mejora
8,j,j_target_enc,0.543508,0.642816,0.099309,True
2,c,c_bin,0.510332,0.540862,0.030529,True
13,score,score_bin,0.725634,0.752185,0.026551,True
4,e,e_bin,0.528351,0.531593,0.003242,True
5,f,f_log,0.686476,0.686894,0.000418,True
0,a,a_freq_enc,0.544588,0.544588,0.000000,False
1,b,b_log,0.561538,0.561538,0.000000,False
3,d,d_log,0.614884,0.614884,0.000000,False
7,h,h_log,0.562323,0.562323,0.000000,False
9,l,l_log,0.675471,0.675471,0.000000,False


**Lectura de esta comparación:**
- Un `delta` positivo significa que la feature nueva separa mejor las clases que la variable original, sola.
- Los encodings de frecuencia/target, suelen ser los que más mejoran sobre variables categóricas de alta cardinalidad (`j`), porque condensan el patrón de fraude por categoría en un solo score continuo — algo que la variable cruda no puede expresar directamente en una prueba univariada.
- Las transformaciones puramente de forma (`_log`) sobre variables ya poco sesgadas, o los flags (`_missing`, `_is_zero`) sobre columnas con muy pocos nulos/ceros, tienden a no mejorar mucho frente al original: no cambian el ordenamiento de las observaciones, solo la escala o resaltan una minoría muy pequeña de filas.
- Esta comparación es univariada (misma limitación que el resto del notebook): que una feature nueva no mejore aquí no significa que sea inútil en un modelo multivariado (p.ej. una interacción o un PCA puede aportar en conjunto con otras variables aunque sola no gane a su original).

**Notas:**
- `o` (categórica) obtiene un IV muy alto (>0.5), consistente con lo ya documentado en el EDA: no es leakage, es la variable más discriminante del dataset por diseño del negocio (el nulo de `o` concentra la señal más fuerte). Aun así, conviene señalarlo explícitamente como corresponde a la interpretación embebida ("sospechoso, revisar leakage") para no asumirlo sin verificar.
- Los `p_value` en 0.0 exacto son producto de redondeo de punto flotante (el valor real es extremadamente pequeño, no exactamente cero) — normal con ~105k filas y señales fuertes.
- La prueba de Wald y el likelihood-ratio test marcan `logit_separation_detected`/`lr_separation_detected=True` (con NaN en el estadístico) cuando el predictor separa casi perfectamente las clases — p.ej. `g` tiene categorías con muy pocas filas (2-3) que caen todas del lado `fraude=0`, lo que vuelve singular la matriz de diseño del modelo con dummies. En esos casos el resto de las pruebas (Mann-Whitney, KS, IV) siguen siendo la referencia más confiable.

## Ranking final: todas las features frente al target

Se unifican las 31 continuas y las 46 categóricas (77 en total, todo lo probado en este notebook) en una sola tabla ordenada por poder predictivo individual, usando la misma métrica común de las secciones anteriores (`auc_abs` para continuas, `auc_woe_abs` para categóricas, ambas en escala 0.5–1, comparables entre sí).

Regla de clasificación (reutiliza los umbrales de AUC ya definidos arriba, no se inventan nuevos):
- **Útil**: `auc_interpretation` es "separación moderada" o "separación fuerte" (auc_abs ≥ 0.7).
- **Marginal**: "separación débil" (0.6 ≤ auc_abs < 0.7) — sola aporta poco, pero puede sumar en un modelo multivariado (interacciones, regularización).
- **Descartable**: "casi aleatorio" (auc_abs < 0.6) — candidata a excluir del modelado.
- **Revisar antes de decidir**: si `logit_separation_detected`/`lr_separation_detected` es `True` (la prueba no es numéricamente confiable, sin importar el AUC), sin importar el bucket anterior.

Adicionalmente se marca con una advertencia (no cambia la clasificación) cuando `iv_interpretation == "sospechoso, revisar leakage"`, ya que un IV muy alto puede ser señal legítima de negocio (como `o`) o una fuga de información, y merece inspección puntual antes de usarse tal cual.

In [12]:
continuous_ranked = stats_fraud_continuos[[
    "variable", "auc_abs", "auc_interpretation", "cohens_d", "cohens_d_interpretation",
    "mannwhitney_p_value", "logit_separation_detected",
]].copy()
continuous_ranked["tipo"] = "continua"
continuous_ranked["effect_size"] = continuous_ranked["cohens_d"].abs()
continuous_ranked["effect_size_interpretation"] = continuous_ranked["cohens_d_interpretation"]
continuous_ranked["iv"] = pd.NA
continuous_ranked["iv_interpretation"] = pd.NA
continuous_ranked["p_value"] = continuous_ranked["mannwhitney_p_value"]
continuous_ranked["separation_detected"] = continuous_ranked["logit_separation_detected"]

categorical_ranked = stats_fraud_categorical[[
    "variable", "auc_woe_abs", "auc_woe_interpretation", "cramers_v", "cramers_v_interpretation",
    "iv", "iv_interpretation", "chi2_p_value", "lr_separation_detected",
]].copy()
categorical_ranked["tipo"] = "categórica"
categorical_ranked = categorical_ranked.rename(columns={
    "auc_woe_abs": "auc_abs",
    "auc_woe_interpretation": "auc_interpretation",
    "cramers_v": "effect_size",
    "cramers_v_interpretation": "effect_size_interpretation",
    "chi2_p_value": "p_value",
    "lr_separation_detected": "separation_detected",
})

common_cols = [
    "variable", "tipo", "auc_abs", "auc_interpretation",
    "effect_size", "effect_size_interpretation",
    "iv", "iv_interpretation", "p_value", "separation_detected",
]
all_features_ranking = pd.concat(
    [continuous_ranked[common_cols], categorical_ranked[common_cols]],
    ignore_index=True,
)


def _clasificar(row):
    if row["separation_detected"]:
        return "Revisar antes de decidir (inestabilidad numérica)"
    if row["auc_interpretation"] == "separación fuerte" or row["auc_interpretation"] == "separación moderada":
        return "Útil"
    if row["auc_interpretation"] == "separación débil":
        return "Marginal (evaluar en conjunto)"
    return "Descartable"


all_features_ranking["clasificacion"] = all_features_ranking.apply(_clasificar, axis=1)
all_features_ranking["advertencia_iv"] = all_features_ranking["iv_interpretation"].eq("sospechoso, revisar leakage")

all_features_ranking = all_features_ranking.sort_values("auc_abs", ascending=False).reset_index(drop=True)
all_features_ranking


,variable,tipo,auc_abs,auc_interpretation,effect_size,effect_size_interpretation,iv,iv_interpretation,p_value,separation_detected,clasificacion,advertencia_iv
0,o_freq_enc,continua,0.755944,separación moderada,1.078816,grande,<NA>,<NA>,0.000000,False,Útil,False
1,o,categórica,0.755944,separación moderada,0.293595,débil,1.114077,"sospechoso, revisar leakage",0.000000,False,Útil,True
2,o_target_enc,continua,0.755146,separación moderada,1.365126,grande,<NA>,<NA>,0.000000,False,Útil,False
3,score_bin,categórica,0.752185,separación moderada,0.231658,débil,0.9588,"sospechoso, revisar leakage",0.000000,False,Útil,True
4,score,continua,0.725634,separación moderada,0.781717,mediano,<NA>,<NA>,0.000000,False,Útil,False
...,...,...,...,...,...,...,...,...,...,...,...,...
72,m_missing,categórica,0.501206,casi aleatorio,0.009617,despreciable,0.001627,inútil,0.001833,False,Descartable,False
73,d_missing,categórica,0.501206,casi aleatorio,0.009617,despreciable,0.001627,inútil,0.001833,False,Descartable,False
74,g_missing,categórica,0.500248,casi aleatorio,0.002796,despreciable,0.000272,inútil,0.364973,False,Descartable,False
75,f_missing,categórica,0.500055,casi aleatorio,0.000295,despreciable,0.000005,inútil,0.923917,True,Revisar antes de decidir (inestabilidad numérica),False


In [13]:
print("Distribución de features por clasificación:")
print(all_features_ranking["clasificacion"].value_counts())

print("\n--- Útiles (mantener) ---")
display(all_features_ranking[all_features_ranking["clasificacion"] == "Útil"][
    ["variable", "tipo", "auc_abs", "auc_interpretation", "iv_interpretation", "advertencia_iv"]
])

print("\n--- Marginales (evaluar en conjunto) ---")
display(all_features_ranking[all_features_ranking["clasificacion"] == "Marginal (evaluar en conjunto)"][
    ["variable", "tipo", "auc_abs", "auc_interpretation"]
])

print("\n--- Descartables (candidatas a excluir) ---")
display(all_features_ranking[all_features_ranking["clasificacion"] == "Descartable"][
    ["variable", "tipo", "auc_abs", "auc_interpretation"]
])

print("\n--- A revisar antes de decidir (inestabilidad numérica) ---")
display(all_features_ranking[all_features_ranking["clasificacion"].str.startswith("Revisar")][
    ["variable", "tipo", "auc_abs", "auc_interpretation"]
])


Distribución de features por clasificación:
clasificacion
Descartable                                          49
Marginal (evaluar en conjunto)                       16
Útil                                                  6
Revisar antes de decidir (inestabilidad numérica)     6
Name: count, dtype: int64

--- Útiles (mantener) ---


,variable,tipo,auc_abs,auc_interpretation,iv_interpretation,advertencia_iv
0,o_freq_enc,continua,0.755944,separación moderada,<NA>,False
1,o,categórica,0.755944,separación moderada,"sospechoso, revisar leakage",True
2,o_target_enc,continua,0.755146,separación moderada,<NA>,False
3,score_bin,categórica,0.752185,separación moderada,"sospechoso, revisar leakage",True
4,score,continua,0.725634,separación moderada,<NA>,False
5,score_log,continua,0.725634,separación moderada,<NA>,False



--- Marginales (evaluar en conjunto) ---


,variable,tipo,auc_abs,auc_interpretation
6,f_log,continua,0.686894,separación débil
7,f,continua,0.686476,separación débil
9,l_log,continua,0.675471,separación débil
10,l,continua,0.675471,separación débil
12,ratio_monto_l,continua,0.660961,separación débil
13,m,continua,0.660395,separación débil
14,m_log,continua,0.660395,separación débil
15,pca_dm_1,continua,0.659319,separación débil
16,m_bin,categórica,0.652740,separación débil
17,j_target_enc,continua,0.642816,separación débil



--- Descartables (candidatas a excluir) ---


,variable,tipo,auc_abs,auc_interpretation
24,m_is_zero,categórica,0.575350,casi aleatorio
26,h_log,continua,0.562323,casi aleatorio
27,h,continua,0.562323,casi aleatorio
28,b_log,continua,0.561538,casi aleatorio
29,b,continua,0.561538,casi aleatorio
30,b_bin,categórica,0.561018,casi aleatorio
31,d_top1pct,categórica,0.559580,casi aleatorio
32,monto,continua,0.559277,casi aleatorio
33,monto_log,continua,0.559277,casi aleatorio
34,h_bin,categórica,0.555124,casi aleatorio



--- A revisar antes de decidir (inestabilidad numérica) ---


,variable,tipo,auc_abs,auc_interpretation
8,f_bin,categórica,0.676986,separación débil
11,l_bin,categórica,0.661030,separación débil
25,j_freq_enc,continua,0.565071,casi aleatorio
43,g,categórica,0.539622,casi aleatorio
75,f_missing,categórica,0.500055,casi aleatorio
76,l_missing,categórica,0.500055,casi aleatorio


In [14]:
all_features_ranking.to_csv(OUTPUT_DIR / "ranking_final_features.csv", index=False)
print(f"Guardado: {OUTPUT_DIR / 'ranking_final_features.csv'}")


Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/stats_outputs/ranking_final_features.csv


**Conclusión general:**
- **Útiles (6/77, AUC ≥ 0.7)**: `o`, `o_freq_enc`, `o_target_enc`, `score`, `score_log`, `score_bin`. Coherente con el EDA: `o` es la señal más fuerte del dataset y `score` ya es un score de riesgo preexistente. Ninguna otra variable, ni siquiera `monto`, llega individualmente a este nivel.
- **Marginales (16/77, AUC 0.6-0.7)**: `f`, `l`, `m`, `d`, `n`, `p`, y varias de sus transformaciones (`f_log`, `l_log`, `m_log`, `m_bin`, `d_log`, `d_bin`, `f_is_zero`), además de `ratio_monto_l`, `pca_dm_1` y `j_target_enc`. No separan bien solas, pero varias ya mostraron mejoras reales en la sección de comparación (`j_target_enc`, `m_bin`, `d_bin`) y son las candidatas más razonables para aportar en un modelo multivariado (interacciones, no linealidades).
- **Descartables (49/77, AUC < 0.6)**: aquí cae la mayoría de los flags (`_missing`, `_is_zero`, `_top1pct`) sobre columnas con pocos nulos/ceros, casi todas las variantes de `fecha` (`fecha_hour`, `fecha_hour_sin/cos`, `fecha_weekday`, `fecha_is_weekend`), `a`/`g`/`j` en su forma original o agrupada, y —de forma menos esperada pero consistente con el EDA (sección de correlación lineal)— también `monto`, `c`, `e`, `h`, `b` y sus transformaciones de forma (`_log`, `_bin`): individualmente aportan poco, aunque puedan ser útiles en combinación con otras variables.
- **A revisar antes de decidir (6/77)**: `f_bin` y `l_bin` tienen AUC prometedor (0.68 y 0.66) pero su regresión logística con dummies no convergió (categorías con muy pocas observaciones) — vale la pena recalcular con menos bins o agrupando más antes de descartarlas. `g`, `f_missing`, `l_missing` y `j_freq_enc` caen en el mismo caso con AUC más bajo.
- Esta tabla es univariada: ninguna clasificación de "Descartable" debería ser definitiva sin confirmar con un chequeo multivariado (p.ej. importancia de un modelo de árboles o regularización L1), pero sí sirve como primer filtro razonable si se necesita reducir dimensionalidad.
- `o`, `o_freq_enc`, `o_target_enc` y `score_bin` quedan además con la advertencia de IV "sospechoso, revisar leakage" (`advertencia_iv=True`): ya se documentó que para `o` es señal de negocio legítima, pero conviene confirmar el origen de `score` (ver la fila correspondiente en la tabla de EDA) antes de asumir lo mismo para `score_bin`.

# Resumen Final

Tabla única con las 77 features evaluadas: qué transformación se aplicó (o si es la variable original sin transformar), qué batería de pruebas se le corrió, su IV y AUC, su nivel de poder predictivo individual, y una observación general que sintetiza todo lo visto en las secciones anteriores.

In [15]:
TRANSFORM_DESCRIPTIONS = {
    "_missing": "Missing indicator (flag de nulo)",
    "_is_zero": "Zero flag (flag de valor cero)",
    "_log": "Transformación logarítmica (log1p)",
    "_top1pct": "Percentile flag (top 1%)",
    "_bin": "Bins por cuantiles (5 bins)",
    "_grouped": "Agrupación de categorías raras (rare grouping)",
    "_freq_enc": "Frequency encoding",
    "_target_enc": "Target encoding con validación cruzada",
}

SPECIAL_TRANSFORM_DESCRIPTIONS = {
    "pca_dm_1": "PCA (1 componente) sobre d y m",
    "ratio_monto_l": "Ratio monto / l",
    "fecha_hour": "Extracción de hora del día",
    "fecha_weekday": "Extracción de día de la semana",
    "fecha_is_weekend": "Extracción de indicador de fin de semana",
    "fecha_hour_sin": "Codificación cíclica de hora (seno)",
    "fecha_hour_cos": "Codificación cíclica de hora (coseno)",
}

RAW_ORIGINAL_COLS = {"a", "b", "c", "d", "e", "f", "g", "h", "j", "l", "m", "n", "o", "p", "monto", "score"}


def _transformacion_aplicada(variable: str) -> str:
    if variable in RAW_ORIGINAL_COLS:
        return "Variable original (sin transformar)"
    if variable in SPECIAL_TRANSFORM_DESCRIPTIONS:
        return SPECIAL_TRANSFORM_DESCRIPTIONS[variable]
    for suffix, desc in TRANSFORM_DESCRIPTIONS.items():
        if variable.endswith(suffix):
            return desc
    return "Transformación no clasificada"


AUC_LABEL_MAP = {
    "casi aleatorio": "casi aleatoria",
    "separación débil": "débil",
    "separación moderada": "moderada",
    "separación fuerte": "fuerte",
}

TIPO_TEST_MAP = {
    "continua": "Continua (Mann-Whitney, KS, punto-biserial, AUC, Cohen\'s d, Wald)",
    "categórica": "Categórica (Chi², V de Cramér, IV/WoE, AUC-WoE, LR test)",
}


def _observaciones(row) -> str:
    partes = [row["clasificacion"]]
    if pd.notna(row["effect_size_interpretation"]):
        etiqueta = "Cohen\'s d" if row["tipo"] == "continua" else "V de Cramér"
        partes.append(f"efecto ({etiqueta}) {row['effect_size_interpretation']}")
    if row["advertencia_iv"]:
        partes.append("IV sospechoso, revisar posible leakage")
    if row["separation_detected"]:
        partes.append("inestabilidad numérica en el ajuste logístico")
    return "; ".join(partes)


resumen_final = all_features_ranking.copy()
resumen_final["transformacion_aplicada"] = resumen_final["variable"].apply(_transformacion_aplicada)
resumen_final["tipo_test_aplicado"] = resumen_final["tipo"].map(TIPO_TEST_MAP)
resumen_final["poder_predictivo"] = resumen_final["auc_interpretation"].map(AUC_LABEL_MAP)
resumen_final["observaciones_generales"] = resumen_final.apply(_observaciones, axis=1)

resumen_final = resumen_final[[
    "variable", "transformacion_aplicada", "tipo_test_aplicado",
    "iv", "auc_abs", "poder_predictivo", "observaciones_generales",
]].sort_values("auc_abs", ascending=False).reset_index(drop=True)



In [16]:
pd.set_option('display.max_rows', 80)
resumen_final

,variable,transformacion_aplicada,tipo_test_aplicado,iv,auc_abs,poder_predictivo,observaciones_generales
0,o_freq_enc,Frequency encoding,"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.755944,moderada,Útil; efecto (Cohen's d) grande
1,o,Variable original (sin transformar),"Categórica (Chi², V de Cramér, IV/WoE, AUC-WoE...",1.114077,0.755944,moderada,Útil; efecto (V de Cramér) débil; IV sospechos...
2,o_target_enc,Target encoding con validación cruzada,"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.755146,moderada,Útil; efecto (Cohen's d) grande
3,score_bin,Bins por cuantiles (5 bins),"Categórica (Chi², V de Cramér, IV/WoE, AUC-WoE...",0.9588,0.752185,moderada,Útil; efecto (V de Cramér) débil; IV sospechos...
4,score,Variable original (sin transformar),"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.725634,moderada,Útil; efecto (Cohen's d) mediano
5,score_log,Transformación logarítmica (log1p),"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.725634,moderada,Útil; efecto (Cohen's d) pequeño
6,f_log,Transformación logarítmica (log1p),"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.686894,débil,Marginal (evaluar en conjunto); efecto (Cohen'...
7,f,Variable original (sin transformar),"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.686476,débil,Marginal (evaluar en conjunto); efecto (Cohen'...
8,f_bin,Bins por cuantiles (5 bins),"Categórica (Chi², V de Cramér, IV/WoE, AUC-WoE...",0.428635,0.676986,débil,Revisar antes de decidir (inestabilidad numéri...
9,l,Variable original (sin transformar),"Continua (Mann-Whitney, KS, punto-biserial, AU...",<NA>,0.675471,débil,Marginal (evaluar en conjunto); efecto (Cohen'...


In [17]:
RESUMEN_FINAL_PATH = OUTPUT_DIR / "resumen_final_features.csv"
resumen_final.to_csv(RESUMEN_FINAL_PATH, index=False)
print(f"Guardado: {RESUMEN_FINAL_PATH}")


Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/stats_outputs/resumen_final_features.csv
